<a href="https://colab.research.google.com/github/SelvaKumaran-G/flyrank-ml-internship3/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SelvaKumaran-G/flyrank-ml-internship3/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# Add to Section 1 — Archetype → action **mapping**

In [2]:
def archetype(row):
    if row['is_declining_label'] == 1 and row['days_since_last_update'] >= 180:
        return 'stale_and_declining'
    elif row['is_declining_label'] == 1:
        return 'declining_fresh'
    elif row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        return 'aging_but_visible'
    elif row['avg_position'] <= 10 and row['ctr'] < 0.5:
        return 'ranked_undercapturing'
    else:
        return 'stable_healthy'

test_df['archetype'] = test_df.apply(archetype, axis=1)
print(test_df['archetype'].value_counts())

## Archetype → action mapping

| Archetype | What it looks like | Mapped action |
|---|---|---|
| stale_and_declining | Declining traffic + 180+ days untouched | **refresh_priority** |
| declining_fresh | Declining despite recent updates | **monitor** — refresh likely isn't the fix; investigate SERP/competitor changes instead |
| aging_but_visible | Stable but old, still gets real impressions | **refresh_review** — proactive, before it turns into a decline |
| ranked_undercapturing | Ranks well (top 10) but low CTR | **review metadata/title**, not full content rewrite |
| stable_healthy | No risk signals | **no_action / protect** |

`declining_fresh` is the archetype most likely to be *wrongly* assigned a
refresh action — freshness didn't cause this decline, so a refresh probably
won't fix it either. This mapping exists specifically to stop that mismatch.

# Add to Section 1 or a new cell — Decay/refresh insight

In [ ]:
# Does staleness actually predict decline, or is it noise?
import numpy as np
bins = [0, 90, 180, 365, np.inf]
labels = ['0-90d', '91-180d', '181-365d', '365d+']
df['staleness_bucket'] = pd.cut(df['days_since_last_update'], bins=bins, labels=labels)
decay_table = df.groupby('staleness_bucket', observed=True).agg(
    n=('content_id','count'),
    decline_rate=('is_declining_label','mean')
)
print(decay_table)

## Decay / refresh insight

Decline rate rises with staleness in this dataset: from `[X]`% at 0-90 days
to `[Y]`% at 365+ days (n shown per bucket above). This is the core insight
behind the playbook's staleness-weighted reason codes — but it's an
**observed association in this snapshot**, not proof that refreshing a stale
page will reverse its decline. A page can be stale *and* declining for
unrelated reasons (e.g. lost backlinks), so staleness earns a page a *look*,
not an automatic refresh.

## Cost / value thinking

- **Cost of a refresh:** writer + editor time (roughly 2-6 hours per page
  depending on scope) — a real, bounded cost per action taken.
- **Cost of a false positive** (refreshed a page that wasn't really declining,
  or was declining for non-content reasons): wasted hours, and a genuinely
  at-risk page goes untouched in its place — an opportunity cost, not just
  wasted effort.
- **Cost of a false negative** (missed page keeps declining unnoticed):
  compounding traffic loss until the next review cycle catches it.
- **Where the model earns its keep:** at Precision@50 ≈ 0.70 (vs. baseline
  ≈ 0.32), roughly 7 of the top 10 flagged pages are genuinely declining —
  meaning refresh hours are spent correctly about twice as often as the
  baseline rule would manage, for the same review capacity.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

df = pd.read_csv('/content/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

numeric = ['search_volume','competition','cpc','word_count','char_count','days_with_impressions',
           'days_since_last_update','ctr','avg_position','engagement_rate','scroll_rate',
           'ai_traffic_pct','content_age_days']
categorical = ['competition_level','content_type','main_intent']

pre = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')),
                       ('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical),
])

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
tr_idx, te_idx = next(splitter.split(df, groups=df['client_id']))
train_df, test_df = df.iloc[tr_idx].copy(), df.iloc[te_idx].copy()

pipe = Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=2000, random_state=42))])
pipe.fit(train_df[numeric+categorical], train_df['is_declining_label'])
test_df['model_score'] = pipe.predict_proba(test_df[numeric+categorical])[:, 1]

def reason(row):
    reasons = []
    if row['days_since_last_update'] >= 180: reasons.append('stale_content')
    if row['avg_position'] > 10: reasons.append('weak_position')
    if row['impressions_90d'] >= 500: reasons.append('meaningful_demand')
    if row['ctr'] < 0.5 and row['impressions_90d'] >= 500: reasons.append('low_ctr_visible')
    return '|'.join(reasons) if reasons else 'general_review'

def action(row):
    if row['model_score'] >= 0.7 and 'stale_content' in row['reason_codes']:
        return 'refresh_priority'
    elif row['model_score'] >= 0.7:
        return 'refresh_review'
    elif row['model_score'] >= 0.4:
        return 'monitor'
    return 'no_action'

test_df['reason_codes'] = test_df.apply(reason, axis=1)
test_df['action_label'] = test_df.apply(action, axis=1)

queue = test_df.sort_values('model_score', ascending=False)[
    ['content_id','client_id','model_score','reason_codes','action_label','is_declining_label']
]
print(queue['action_label'].value_counts())
queue.head(10)

action_label
monitor           4412
no_action         1131
refresh_review     620
Name: count, dtype: int64


,content_id,client_id,model_score,reason_codes,action_label,is_declining_label
1537,content_a928cb66d230,client_f369cb89fc,0.928925,general_review,refresh_review,1
10175,content_374e795aab68,client_f369cb89fc,0.918802,weak_position,refresh_review,0
3626,content_8ede62882d0b,client_f369cb89fc,0.907642,weak_position|meaningful_demand|low_ctr_visible,refresh_review,1
605,content_5d77d3077984,client_f369cb89fc,0.899665,general_review,refresh_review,1
24766,content_4315e2ea3148,client_f369cb89fc,0.894369,general_review,refresh_review,1
26614,content_7be5f150dc65,client_f369cb89fc,0.892398,general_review,refresh_review,0
8688,content_35fb876ad42a,client_f369cb89fc,0.888097,general_review,refresh_review,1
29037,content_94defa9a6783,client_f369cb89fc,0.887447,general_review,refresh_review,1
14741,content_2bc3b7c8b3d9,client_f369cb89fc,0.886345,meaningful_demand,refresh_review,1
3195,content_b5e9e6453511,client_f369cb89fc,0.885983,general_review,refresh_review,1


## Ranked actions

The queue ranks pages by predicted decline risk (`model_score`), then assigns
one of four actions:
- **refresh_priority** — high score AND stale content (180+ days untouched):
  the clearest combined signal, do these first
- **refresh_review** — high score, but not driven by staleness alone: worth a
  human look before committing writer time
- **monitor** — moderate score: track next cycle, no action yet
- **no_action** — low score: leave as-is

Each row also carries plain reason codes (`stale_content`, `weak_position`,
`meaningful_demand`, `low_ctr_visible`) so a reviewer can see *why* a page
ranked where it did, not just a number.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## Intended use

**Who uses this:** A content strategist or SEO lead doing monthly refresh
planning, deciding where to spend limited writer/editor hours.

**What it's for:** Prioritizing *review*, not replacing judgment — it narrows
30,000+ pages down to a short, ranked shortlist worth a human look.

**Where it stops being valid:**
- This model was trained and validated on one anonymized portfolio snapshot
  (90-day window). It has not been tested on other time periods or other
  content verticals.
- It predicts *observed* decline risk, not the cause of decline — a page can
  score high because of staleness, a lost featured snippet, seasonality, or a
  competitor's new content, and the model can't tell these apart.
- It is decision-support only. It ranks and flags at a measured Precision@50
  of ~0.70 under a client-grouped holdout split — it does not predict or
  reverse-engineer Google's ranking algorithm, and it does not guarantee any
  individual page's outcome.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
review_checklist = """
Before acting on any 'refresh_priority' or 'refresh_review' row, a human should check:
1. Is the decline seasonal or industry-wide (check a broader trend/competitor context)?
2. Has this page recently lost a SERP feature (featured snippet, PAA) unrelated to content quality?
3. Is this page still strategically relevant to the business (not a candidate for pruning instead)?
4. Does the 'reason_codes' story actually match what a human sees on the page?
"""
print(review_checklist)


Before acting on any 'refresh_priority' or 'refresh_review' row, a human should check:
1. Is the decline seasonal or industry-wide (check a broader trend/competitor context)?
2. Has this page recently lost a SERP feature (featured snippet, PAA) unrelated to content quality?
3. Is this page still strategically relevant to the business (not a candidate for pruning instead)?
4. Does the 'reason_codes' story actually match what a human sees on the page?



## No-go list — what should NOT be automated

- **Never auto-publish or auto-edit content** based on this score alone — every
  action requires human sign-off.
- **Never use this to auto-deprioritize or hide pages** from search/sitemaps —
  a "no_action" label means "not flagged this cycle," not "safe to remove."
- **Never treat the model score as a client-facing guarantee** of traffic
  recovery if refreshed.
- **Never skip human review for pages with thin reason codes** (`general_review`)
  — these are the least-explained, highest-uncertainty cases and need the most
  scrutiny, not the least.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## Monitoring / retrain triggers

Signals that would mean this playbook has gone stale and needs a fresh look:
- **Precision@50 drift:** if a re-evaluation on a newer data pull drops
  meaningfully below the ~0.70 baseline measured in Week 6, the model may no
  longer reflect current patterns.
- **Feature distribution shift:** if average `days_since_last_update` or
  `avg_position` in new data looks very different from the training snapshot,
  the model is extrapolating outside what it learned from.
- **Action volume shift:** if the share of pages flagged `refresh_priority`
  jumps or collapses sharply between runs, investigate before trusting the
  new queue.
- **Time-based trigger:** re-validate at least quarterly, since this was
  trained on a single 90-day snapshot and content/search behavior changes
  over time.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
import os

os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/refresh_action_queue.csv', index=False)
print(f"Wrote {len(queue)} rows to work/outputs/refresh_action_queue.csv")

# summary metrics JSON — committed, not gitignored, per GUIDE.md
import json
summary = {
    "precision_at_50_grouped_split": 0.700,
    "precision_at_50_baseline": 0.32,
    "base_rate": float(df['is_declining_label'].mean()),
    "action_counts": queue['action_label'].value_counts().to_dict(),
    "split_strategy": "client_holdout_group_shuffle",
    "random_seed": 42,
}
with open('work/outputs/w07_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("Wrote work/outputs/w07_summary.json")

Wrote 6163 rows to work/outputs/refresh_action_queue.csv
Wrote work/outputs/w07_summary.json


Exported `work/outputs/refresh_action_queue.csv` (the ranked queue with reason
codes and actions) and `work/outputs/w07_summary.json` (run metrics) — both
feed directly into next week's Results and Recommendations sections.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.